# Table 6 대안모형 및 강건성 분석 결과표 생성 노트북

## 1. 목적

이 노트북은 2024년 기업정보화통계조사 최종 분석용 데이터셋을 기준으로, 중간보고서 4. 예비분석 결과 섹션에 들어갈 대안모형 및 강건성 분석 결과표를 생성한다.

이번 단계의 핵심 목적은 주모형 결과 이후 보조모형/대안모형을 통해 핵심 계수의 방향과 유의성이 대체로 유지되는지 확인하는 것이다. 모든 계수를 자세히 해석하기보다 핵심 계수의 방향, 유의성, 적합도, 주모형과의 일치 여부를 중심으로 정리한다.

## 2. 설정값 및 저장 옵션

`SAVE_OUTPUTS` 플래그 하나로 결과 파일 저장 여부를 관리한다.

- `SAVE_OUTPUTS = True`: 결과표 CSV/Markdown과 해석 메모를 `output/tables`에 저장한다.
- `SAVE_OUTPUTS = False`: 파일 저장 없이 노트북 화면에만 결과를 표시한다.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel

try:
    from IPython.display import display
except ImportError:
    display = None

# 저장 여부 단일 플래그
SAVE_OUTPUTS = False

CWD = Path.cwd()
if (CWD / 'working').exists():
    BASE_DIR = CWD
elif (CWD.parent / 'working').exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError('working 폴더가 있는 프로젝트 루트를 찾지 못했습니다.')

ANALYSIS_DIR = BASE_DIR / 'working' / 'analysis'
OUT_DIR = BASE_DIR / 'output' / 'tables'
if SAVE_OUTPUTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('OUT_DIR:', OUT_DIR)
print('SAVE_OUTPUTS:', SAVE_OUTPUTS)

## 3. 데이터 로드 및 변수 확인

최종 분석용 데이터셋은 `working/analysis` 폴더에서만 찾는다. 필요한 변수 존재 여부와 결측 N을 먼저 확인한다.

In [ ]:
patterns = ['*.csv', '*.xlsx', '*.parquet']
candidates = []
for priority, pattern in enumerate(patterns, start=1):
    for path in ANALYSIS_DIR.glob(pattern):
        candidates.append({
            'priority': priority,
            'file': path.name,
            'path': path,
            'modified_time': pd.Timestamp(path.stat().st_mtime, unit='s'),
            'size_bytes': path.stat().st_size,
        })

candidate_df = pd.DataFrame(candidates).sort_values(['priority', 'modified_time'], ascending=[True, False])
if candidate_df.empty:
    raise FileNotFoundError('working/analysis 안에서 csv/xlsx/parquet 분석 파일을 찾지 못했습니다.')

DATA_PATH = candidate_df.iloc[0]['path']
print('[working/analysis 데이터 후보]')
if display:
    display(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']])
else:
    print(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']].to_string(index=False))
print('\n사용 데이터 파일:', DATA_PATH.relative_to(BASE_DIR))

suffix = DATA_PATH.suffix.lower()
if suffix == '.csv':
    df = pd.read_csv(DATA_PATH)
elif suffix in ['.xlsx', '.xls']:
    df = pd.read_excel(DATA_PATH)
elif suffix == '.parquet':
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError(f'지원하지 않는 파일 형식입니다: {suffix}')

total_n = len(df)
required_vars = ['effect_proc_improve', 'effect_average', 'it_org_any', 'ai_use_sum', 'firm_size', 'industry']
variable_check = pd.DataFrame([
    {
        '변수명': var,
        '존재 여부': var in df.columns,
        '유효 N': int(df[var].notna().sum()) if var in df.columns else np.nan,
        '결측 N': int(df[var].isna().sum()) if var in df.columns else np.nan,
    }
    for var in required_vars
])
missing_vars = variable_check.loc[~variable_check['존재 여부'], '변수명'].tolist()

print('전체 N:', f'{total_n:,}')
print('\n[변수 존재 여부 및 결측 확인]')
if display:
    display(variable_check)
else:
    print(variable_check.to_string(index=False))

if missing_vars:
    raise KeyError('최종 분석용 데이터셋에 없는 변수: ' + ', '.join(missing_vars))

## 4. 대안모형 및 강건성 분석 설계

이번 노트북에서는 다음 여섯 개 모형을 추정한다.

- Alternative Model 1: H1 Ordered Logit
- Alternative Model 2: H3 Ordered Logit with Interaction
- Alternative Model 3: H2 Poisson
- Alternative Model 4: H2 Negative Binomial
- Robustness Model 1: H1 with `effect_average` OLS + HC3
- Robustness Model 2: H3 with `effect_average` OLS + HC3

주모형 비교 기준은 H1, H2, H3 모두 양(+) 방향이며 p < .05로 유의하다는 것이다.

In [ ]:
term_labels = {
    'it_org_any': '정보화 담당 체계 보유',
    'ai_use_sum': 'AI 활용 수준',
    'it_org_any:ai_use_sum': '정보화 담당 체계 × AI 활용 수준',
}
outcome_labels = {
    'effect_proc_improve': '프로세스 개선 효과',
    'effect_average': '정보화 효과 평균',
    'ai_use_sum': 'AI 활용 수준',
}

main_reference = {
    'H1': {'term': 'it_org_any', 'direction': '+', 'significant': True},
    'H2': {'term': 'it_org_any', 'direction': '+', 'significant': True},
    'H3': {'term': 'it_org_any:ai_use_sum', 'direction': '+', 'significant': True},
}

model_specs = [
    {
        'model_name': 'Alternative Model 1',
        'hypothesis': 'H1',
        'model_type': 'Ordered Logit',
        'outcome': 'effect_proc_improve',
        'key_term': 'it_org_any',
        'terms': ['it_org_any'],
        'variables': ['effect_proc_improve', 'it_org_any', 'firm_size', 'industry'],
        'note': '순서형 DV 반영',
    },
    {
        'model_name': 'Alternative Model 2',
        'hypothesis': 'H3',
        'model_type': 'Ordered Logit',
        'outcome': 'effect_proc_improve',
        'key_term': 'it_org_any:ai_use_sum',
        'terms': ['it_org_any', 'ai_use_sum', 'it_org_any:ai_use_sum'],
        'variables': ['effect_proc_improve', 'it_org_any', 'ai_use_sum', 'firm_size', 'industry'],
        'note': '순서형 DV와 상호작용 반영',
    },
    {
        'model_name': 'Alternative Model 3',
        'hypothesis': 'H2',
        'model_type': 'Poisson',
        'outcome': 'ai_use_sum',
        'key_term': 'it_org_any',
        'terms': ['it_org_any'],
        'formula': 'ai_use_sum ~ it_org_any + C(firm_size) + C(industry)',
        'variables': ['ai_use_sum', 'it_org_any', 'firm_size', 'industry'],
        'note': '카운트형 DV 반영',
    },
    {
        'model_name': 'Alternative Model 4',
        'hypothesis': 'H2',
        'model_type': 'Negative Binomial',
        'outcome': 'ai_use_sum',
        'key_term': 'it_org_any',
        'terms': ['it_org_any'],
        'formula': 'ai_use_sum ~ it_org_any + C(firm_size) + C(industry)',
        'variables': ['ai_use_sum', 'it_org_any', 'firm_size', 'industry'],
        'note': '과분산 가능성 반영',
    },
    {
        'model_name': 'Robustness Model 1',
        'hypothesis': 'H1',
        'model_type': 'OLS HC3',
        'outcome': 'effect_average',
        'key_term': 'it_org_any',
        'terms': ['it_org_any'],
        'formula': 'effect_average ~ it_org_any + C(firm_size) + C(industry)',
        'variables': ['effect_average', 'it_org_any', 'firm_size', 'industry'],
        'note': '대체 DV effect_average',
    },
    {
        'model_name': 'Robustness Model 2',
        'hypothesis': 'H3',
        'model_type': 'OLS HC3',
        'outcome': 'effect_average',
        'key_term': 'it_org_any:ai_use_sum',
        'terms': ['it_org_any', 'ai_use_sum', 'it_org_any:ai_use_sum'],
        'formula': 'effect_average ~ it_org_any * ai_use_sum + C(firm_size) + C(industry)',
        'variables': ['effect_average', 'it_org_any', 'ai_use_sum', 'firm_size', 'industry'],
        'note': '대체 DV effect_average와 상호작용',
    },
]

design_summary = pd.DataFrame([
    {
        'model_name': spec['model_name'],
        'hypothesis': spec['hypothesis'],
        'model_type': spec['model_type'],
        'outcome': spec['outcome'],
        'key_term': spec['key_term'],
        'note': spec['note'],
    }
    for spec in model_specs
])
print('[대안모형/강건성 분석 설계 요약]')
if display:
    display(design_summary)
else:
    print(design_summary.to_string(index=False))

model_n_check = []
for spec in model_specs:
    sub = df[spec['variables']].dropna()
    model_n_check.append({
        'model_name': spec['model_name'],
        '사용 N': len(sub),
        '전체 N 대비 제외 N': total_n - len(sub),
        '결측 변수별 N': {var: int(df[var].isna().sum()) for var in spec['variables']},
    })
model_n_check_df = pd.DataFrame(model_n_check)
print('\n[모형별 사용 N 확인]')
if display:
    display(model_n_check_df)
else:
    print(model_n_check_df.to_string(index=False))

## 5. H1/H3 대안모형: Ordered Logit

`effect_proc_improve`가 1~5점 순서형 변수라는 점을 반영해 Ordered Logit을 추정한다. `statsmodels`의 `OrderedModel`은 상수항을 포함하지 않으므로, `firm_size`와 `industry`는 기준범주를 제외한 더미변수로 변환한다.

In [ ]:
def make_ordered_data(data, outcome, include_ai=False, include_interaction=False):
    cols = [outcome, 'it_org_any', 'firm_size', 'industry']
    if include_ai:
        cols.append('ai_use_sum')
    model_data = data[cols].dropna().copy()
    y = model_data[outcome].astype(int)
    x_parts = [model_data[['it_org_any']].astype(float)]
    if include_ai:
        x_parts.append(model_data[['ai_use_sum']].astype(float))
    if include_interaction:
        interaction = (model_data['it_org_any'] * model_data['ai_use_sum']).astype(float).rename('it_org_any:ai_use_sum')
        x_parts.append(interaction.to_frame())
    firm_dummies = pd.get_dummies(model_data['firm_size'].astype(int), prefix='firm_size', drop_first=True, dtype=float)
    industry_dummies = pd.get_dummies(model_data['industry'].astype(int), prefix='industry', drop_first=True, dtype=float)
    x_parts.extend([firm_dummies, industry_dummies])
    X = pd.concat(x_parts, axis=1).astype(float)
    return y, X, model_data

ordered_results = {}
ordered_errors = {}

for spec in [s for s in model_specs if s['model_type'] == 'Ordered Logit']:
    try:
        include_ai = 'ai_use_sum' in spec['variables']
        include_interaction = spec['key_term'] == 'it_org_any:ai_use_sum'
        y, X, model_data = make_ordered_data(df, spec['outcome'], include_ai=include_ai, include_interaction=include_interaction)
        model = OrderedModel(y, X, distr='logit')
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            result = model.fit(method='bfgs', disp=False, maxiter=300)
        ordered_results[spec['model_name']] = {'result': result, 'n': len(model_data), 'warnings': [str(w.message) for w in caught]}
        print(f'\n===== {spec["model_name"]}: {spec["hypothesis"]} Ordered Logit =====')
        print(result.summary())
        if caught:
            print('[Warnings]')
            for w in caught:
                print('-', str(w.message))
    except Exception as exc:
        ordered_errors[spec['model_name']] = repr(exc)
        print(f'Ordered Logit 오류 발생 - {spec["model_name"]}: {repr(exc)}')

## 6. H2 대안모형: Count Model

`ai_use_sum`이 0 이상의 카운트 변수라는 점을 반영해 Poisson과 Negative Binomial을 추정한다. 가능하면 robust standard error를 사용한다.

In [ ]:
count_results = {}
count_errors = {}

for spec in [s for s in model_specs if s['model_type'] in ['Poisson', 'Negative Binomial']]:
    try:
        model_data = df[spec['variables']].dropna().copy()
        if spec['model_type'] == 'Poisson':
            model = smf.glm(spec['formula'], data=model_data, family=sm.families.Poisson())
            try:
                result = model.fit(cov_type='HC3')
                se_type = 'HC3 robust SE'
            except Exception:
                result = model.fit(cov_type='HC0')
                se_type = 'HC0 robust SE fallback'
        else:
            model = smf.negativebinomial(spec['formula'], data=model_data)
            try:
                result = model.fit(disp=False, maxiter=300, cov_type='HC3')
                se_type = 'HC3 robust SE'
            except Exception:
                result = model.fit(disp=False, maxiter=300)
                se_type = 'model SE fallback'
        count_results[spec['model_name']] = {'result': result, 'n': len(model_data), 'se_type': se_type}
        print(f'\n===== {spec["model_name"]}: {spec["model_type"]} =====')
        print('SE type:', se_type)
        print(result.summary())
    except Exception as exc:
        count_errors[spec['model_name']] = repr(exc)
        print(f'Count model 오류 발생 - {spec["model_name"]}: {repr(exc)}')

## 7. 강건성 분석: 대체 DV effect_average

메인 DV 대신 `effect_average`를 사용해 H1과 H3 방향이 유지되는지 OLS + HC3 robust standard error로 확인한다.

In [ ]:
ols_results = {}
ols_errors = {}

for spec in [s for s in model_specs if s['model_type'] == 'OLS HC3']:
    try:
        model_data = df[spec['variables']].dropna().copy()
        result = smf.ols(spec['formula'], data=model_data).fit(cov_type='HC3')
        ols_results[spec['model_name']] = {'result': result, 'n': len(model_data), 'se_type': 'HC3 robust SE'}
        print(f'\n===== {spec["model_name"]}: {spec["outcome"]} OLS HC3 =====')
        print(result.summary())
    except Exception as exc:
        ols_errors[spec['model_name']] = repr(exc)
        print(f'OLS 강건성 모형 오류 발생 - {spec["model_name"]}: {repr(exc)}')

## 8. 대안모형/강건성 비교표

보고서용 비교표는 각 모형의 핵심 계수만 포함한다. 별도의 tidy table에는 요청된 핵심 term을 모두 포함한다.

In [ ]:
def stars(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''


def p_display(p):
    if pd.isna(p):
        return 'NA'
    if p < 0.001:
        return '<.001'
    return f'{p:.3f}'


def md_table(data: pd.DataFrame) -> str:
    def fmt(value):
        if pd.isna(value):
            return 'NA'
        if isinstance(value, (float, np.floating)):
            return f'{float(value):.3f}'
        return str(value).replace('|', '\\|').replace('\n', '<br>')
    cols = list(data.columns)
    lines = ['| ' + ' | '.join(cols) + ' |']
    lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
    for _, row in data.iterrows():
        lines.append('| ' + ' | '.join(fmt(row[col]) for col in cols) + ' |')
    return '\n'.join(lines)


def save_table(data: pd.DataFrame, path: Path, **kwargs):
    if SAVE_OUTPUTS:
        data.to_csv(path, **kwargs)
    return path


def save_text(text: str, path: Path):
    if SAVE_OUTPUTS:
        path.write_text(text, encoding='utf-8')
    return path


def print_save_status(path: Path):
    if SAVE_OUTPUTS:
        print('-', path.relative_to(BASE_DIR))
    else:
        print(f'- {path.name}: 저장 비활성화, 노트북 출력만 확인')


def fit_metrics(result, model_type):
    if model_type == 'OLS HC3':
        return 'R-squared', float(result.rsquared), 'Adjusted R-squared', float(result.rsquared_adj)
    llf = getattr(result, 'llf', np.nan)
    aic = getattr(result, 'aic', np.nan)
    return 'Log-likelihood', float(llf) if not pd.isna(llf) else np.nan, 'AIC', float(aic) if not pd.isna(aic) else np.nan


def extract_result(spec):
    name = spec['model_name']
    if spec['model_type'] == 'Ordered Logit':
        if name in ordered_results:
            return ordered_results[name]['result'], ordered_results[name]['n'], 'model SE', None
        return None, None, None, ordered_errors.get(name, 'Ordered Logit result not found')
    if spec['model_type'] in ['Poisson', 'Negative Binomial']:
        if name in count_results:
            return count_results[name]['result'], count_results[name]['n'], count_results[name]['se_type'], None
        return None, None, None, count_errors.get(name, 'Count model result not found')
    if spec['model_type'] == 'OLS HC3':
        if name in ols_results:
            return ols_results[name]['result'], ols_results[name]['n'], ols_results[name]['se_type'], None
        return None, None, None, ols_errors.get(name, 'OLS result not found')
    return None, None, None, 'Unknown model type'


def term_stats(result, term):
    if result is None or term not in result.params.index:
        return {k: np.nan for k in ['coef', 'se', 'test_stat', 'p_value', 'ci_lower', 'ci_upper']}
    conf = result.conf_int(alpha=0.05)
    if isinstance(conf, np.ndarray):
        idx = list(result.params.index).index(term)
        ci_lower, ci_upper = conf[idx, 0], conf[idx, 1]
    else:
        ci_lower, ci_upper = conf.loc[term, 0], conf.loc[term, 1]
    return {
        'coef': float(result.params[term]),
        'se': float(result.bse[term]),
        'test_stat': float(result.tvalues[term]) if hasattr(result, 'tvalues') else np.nan,
        'p_value': float(result.pvalues[term]),
        'ci_lower': float(ci_lower),
        'ci_upper': float(ci_upper),
    }


def direction_consistency(coef, hypothesis):
    if pd.isna(coef):
        return '확인 필요'
    expected_positive = main_reference[hypothesis]['direction'] == '+'
    return '일치' if (coef > 0 and expected_positive) or (coef < 0 and not expected_positive) else '불일치'


def significance_consistency(p_value, hypothesis):
    if pd.isna(p_value):
        return '확인 필요'
    main_sig = main_reference[hypothesis]['significant']
    alt_sig = p_value < 0.05
    return '일치' if main_sig == alt_sig else '불일치'


def brief_note(spec, coef, p_value, failure=None):
    if failure:
        return '수렴 실패 또는 추정 실패: ' + failure
    direction = '양(+)' if coef > 0 else '음(-)' if coef < 0 else '0'
    sig_text = '유의' if p_value < 0.05 else '비유의'
    return f'핵심 계수는 {direction} 방향이며 {sig_text}; 주모형과의 방향/유의성 비교 중심으로 해석'

In [ ]:
# brief_note 함수의 문자열 오탈자를 피하기 위해 재정의한다.
def brief_note(spec, coef, p_value, failure=None):
    if failure:
        return '수렴 실패 또는 추정 실패: ' + failure
    direction = '양(+)' if coef > 0 else '음(-)' if coef < 0 else '0'
    sig_text = '유의' if p_value < 0.05 else '비유의'
    return f'핵심 계수는 {direction} 방향이며 {sig_text}; 주모형과의 방향/유의성 비교 중심으로 해석'

comparison_rows = []
tidy_rows = []

for spec in model_specs:
    result, nobs, se_type, failure = extract_result(spec)
    fit1_name, fit1_value, fit2_name, fit2_value = (np.nan, np.nan, np.nan, np.nan) if result is None else fit_metrics(result, spec['model_type'])
    key_stats = term_stats(result, spec['key_term'])
    direction_match = direction_consistency(key_stats['coef'], spec['hypothesis'])
    sig_match = significance_consistency(key_stats['p_value'], spec['hypothesis'])
    comparison_rows.append({
        '분석 대상 가설': spec['hypothesis'],
        '모형 구분': spec['model_type'],
        '종속변수': outcome_labels.get(spec['outcome'], spec['outcome']),
        '핵심 변수': term_labels.get(spec['key_term'], spec['key_term']),
        '계수': key_stats['coef'],
        'Robust SE 또는 SE': key_stats['se'],
        'p-value': key_stats['p_value'],
        '유의성': stars(key_stats['p_value']),
        'N': nobs if nobs is not None else np.nan,
        '적합도': '확인 필요' if result is None else f'{fit1_name}={fit1_value:.3f}; {fit2_name}={fit2_value:.3f}',
        '주모형과 방향 일치 여부': direction_match,
        '주모형과 유의성 일치 여부': sig_match,
        '해석 메모': brief_note(spec, key_stats['coef'], key_stats['p_value'], failure=failure),
    })
    for term in spec['terms']:
        stats = term_stats(result, term)
        tidy_rows.append({
            'model_name': spec['model_name'],
            'hypothesis': spec['hypothesis'],
            'model_type': spec['model_type'],
            'outcome': spec['outcome'],
            'term': term,
            'coef': stats['coef'],
            'se': stats['se'],
            'test_stat': stats['test_stat'],
            'p_value': stats['p_value'],
            'significance': stars(stats['p_value']),
            'ci_lower': stats['ci_lower'],
            'ci_upper': stats['ci_upper'],
            'n': nobs if nobs is not None else np.nan,
            'fit_metric_1_name': fit1_name,
            'fit_metric_1_value': fit1_value,
            'fit_metric_2_name': fit2_name,
            'fit_metric_2_value': fit2_value,
            'direction_consistent_with_main': direction_consistency(stats['coef'], spec['hypothesis']) if term == main_reference[spec['hypothesis']]['term'] else '',
            'significance_consistent_with_main': significance_consistency(stats['p_value'], spec['hypothesis']) if term == main_reference[spec['hypothesis']]['term'] else '',
            'failure_reason': failure if failure else '',
        })

comparison_table = pd.DataFrame(comparison_rows)
tidy_table = pd.DataFrame(tidy_rows)

for col in ['계수', 'Robust SE 또는 SE', 'p-value']:
    comparison_table[col] = comparison_table[col].astype(float).round(3)
for col in ['coef', 'se', 'test_stat', 'p_value', 'ci_lower', 'ci_upper', 'fit_metric_1_value', 'fit_metric_2_value']:
    tidy_table[col] = tidy_table[col].astype(float).round(3)

comparison_display = comparison_table.copy()
comparison_display['p-value'] = comparison_display['p-value'].apply(p_display)

tidy_display = tidy_table.copy()
tidy_display['p_value'] = tidy_display['p_value'].apply(p_display)

print('Table 6. 대안모형 및 강건성 비교표')
if display:
    display(comparison_display)
else:
    print(comparison_display.to_string(index=False))

print('\n핵심 계수 tidy table')
if display:
    display(tidy_display)
else:
    print(tidy_display.to_string(index=False))

## 9. 보고서용 해석 메모

대안모형과 강건성 분석에서 핵심 계수의 방향과 유의성이 주모형과 대체로 일치하는지 요약한다. 단면자료 기반 분석이므로 인과 표현은 피한다.

In [ ]:
def row_for(model_name):
    return tidy_table.loc[tidy_table['model_name'] == model_name]

def coef_for(model_name, term):
    rows = tidy_table.loc[(tidy_table['model_name'] == model_name) & (tidy_table['term'] == term)]
    return rows.iloc[0] if len(rows) else None

h1_ord = coef_for('Alternative Model 1', 'it_org_any')
h3_ord = coef_for('Alternative Model 2', 'it_org_any:ai_use_sum')
h2_pois = coef_for('Alternative Model 3', 'it_org_any')
h2_nb = coef_for('Alternative Model 4', 'it_org_any')
h1_rob = coef_for('Robustness Model 1', 'it_org_any')
h3_rob = coef_for('Robustness Model 2', 'it_org_any:ai_use_sum')

sentences = []
if h1_ord is not None and not pd.isna(h1_ord['coef']):
    sentences.append(f'H1에 대한 Ordered Logit 결과에서도 정보화 담당 체계 보유 계수는 {h1_ord["coef"]:.3f}{h1_ord["significance"]}로 나타나 주모형과 동일한 양(+)의 방향을 보였다.')
else:
    sentences.append('H1 Ordered Logit은 수렴 실패 또는 추정 실패로 핵심 계수 확인이 필요하다.')

if h2_pois is not None and h2_nb is not None and not pd.isna(h2_pois['coef']) and not pd.isna(h2_nb['coef']):
    sentences.append(f'H2에 대한 Poisson 및 Negative Binomial 결과에서도 정보화 담당 체계 보유 계수는 각각 {h2_pois["coef"]:.3f}{h2_pois["significance"]}, {h2_nb["coef"]:.3f}{h2_nb["significance"]}로 나타나 주모형과 같은 양(+)의 방향을 보였다.')
else:
    sentences.append('H2 카운트 모형 중 일부는 수렴 실패 또는 추정 실패로 결과 확인이 필요하다.')

if h3_ord is not None and not pd.isna(h3_ord['coef']):
    h3_text = '일치하였다' if h3_ord['coef'] > 0 else '일치하지 않았다'
    sentences.append(f'H3에 대한 Ordered Logit 결과에서 상호작용항 계수는 {h3_ord["coef"]:.3f}{h3_ord["significance"]}로 나타나 주모형의 조절효과 방향과 {h3_text}.')
else:
    sentences.append('H3 Ordered Logit은 수렴 실패 또는 추정 실패로 상호작용항 확인이 필요하다.')

if h1_rob is not None and h3_rob is not None and not pd.isna(h1_rob['coef']) and not pd.isna(h3_rob['coef']):
    robustness_text = '유지되었다' if h1_rob['coef'] > 0 and h3_rob['coef'] > 0 else '완전히 유지되지는 않았다'
    sentences.append(f'대체 종속변수인 effect_average를 사용한 강건성 분석에서도 H1 핵심 계수는 {h1_rob["coef"]:.3f}{h1_rob["significance"]}, H3 상호작용항은 {h3_rob["coef"]:.3f}{h3_rob["significance"]}로 나타나 핵심 계수의 방향은 {robustness_text}.')
else:
    sentences.append('effect_average 강건성 분석 중 일부 결과는 확인이 필요하다.')

consistent_direction_count = (comparison_table['주모형과 방향 일치 여부'] == '일치').sum()
consistent_sig_count = (comparison_table['주모형과 유의성 일치 여부'] == '일치').sum()
total_models = len(comparison_table)
sentences.append(f'전체 {total_models}개 대안모형/강건성 모형 중 방향 일치는 {consistent_direction_count}개, 유의성 일치는 {consistent_sig_count}개로 나타나 주모형의 기본 결론이 모형 선택과 종속변수 대체에 대해 대체로 유지되는지 확인할 수 있다.')
sentences.append('다만 본 분석은 단면자료 기반의 예비 분석이므로, 결과를 인과관계로 해석하기에는 제한이 있다.')

print('보고서용 해석 메모')
for s in sentences:
    print('-', s)

## 10. 확인 필요 사항

저장 여부, 저장 파일 목록, 모형별 사용 N, 실패 모형 여부를 확인한다.

In [ ]:
comparison_csv = OUT_DIR / 'table6_alternative_robustness_comparison.csv'
comparison_md = OUT_DIR / 'table6_alternative_robustness_comparison.md'
tidy_csv = OUT_DIR / 'table6_alternative_robustness_tidy.csv'
interp_md = OUT_DIR / 'alternative_robustness_interpretation.md'

save_table(comparison_table, comparison_csv, index=False, encoding='utf-8-sig')
save_table(tidy_table, tidy_csv, index=False, encoding='utf-8-sig')
save_text('# Table 6. 대안모형 및 강건성 비교표\n\n' + md_table(comparison_display) + '\n', comparison_md)
save_text('# 대안모형 및 강건성 분석 해석 메모\n\n' + f'- 사용 데이터: `{DATA_PATH.relative_to(BASE_DIR)}`\n- 전체 N: {total_n:,}\n\n' + '\n'.join(f'- {s}' for s in sentences) + '\n', interp_md)

need_check = []
if missing_vars:
    need_check.append('최종 분석용 데이터셋에 없는 변수: ' + ', '.join(missing_vars))
else:
    need_check.append('필수 변수 6개가 모두 존재함')

if all(row['전체 N 대비 제외 N'] == 0 for _, row in model_n_check_df.iterrows()):
    need_check.append('모든 대안모형/강건성 모형에서 사용 N이 전체 N과 동일하며, 결측으로 인한 표본 손실 없음')
else:
    for _, row in model_n_check_df.iterrows():
        if row['전체 N 대비 제외 N'] > 0:
            need_check.append(f"{row['model_name']}에서 전체 N 대비 {row['전체 N 대비 제외 N']}건 제외됨: {row['결측 변수별 N']}")

all_errors = {**ordered_errors, **count_errors, **ols_errors}
if all_errors:
    for model_name, error in all_errors.items():
        need_check.append(f'{model_name} 추정 확인 필요: {error}')
else:
    need_check.append('모든 대안모형/강건성 모형이 오류 없이 추정됨')
need_check.append('Ordered Logit의 계수는 잠재 순서형 응답 기준의 log-odds 방향으로 해석해야 함')
need_check.append('대안모형 결과는 주모형과의 방향/유의성 비교를 위한 예비 확인용임')

print('1. 사용한 데이터 파일명과 전체 N')
print('-', DATA_PATH.relative_to(BASE_DIR))
print('-', f'N = {total_n:,}')
print('SAVE_OUTPUTS:', SAVE_OUTPUTS)

print('\n2. 변수 존재 여부 및 결측 확인표')
if display:
    display(variable_check)
else:
    print(variable_check.to_string(index=False))

print('\n3. 대안모형/강건성 분석 설계 요약')
if display:
    display(design_summary)
else:
    print(design_summary.to_string(index=False))

print('\n5. Table 6. 대안모형 및 강건성 비교표')
if display:
    display(comparison_display)
else:
    print(comparison_display.to_string(index=False))

print('\n6. 핵심 계수 tidy table')
if display:
    display(tidy_display)
else:
    print(tidy_display.to_string(index=False))

print('\n7. 보고서용 해석 메모')
for s in sentences:
    print('-', s)

print('\n8. 저장 여부 및 저장 파일 목록')
if SAVE_OUTPUTS:
    print('저장 완료')
else:
    print('저장 비활성화: 파일은 생성하지 않고 노트북 화면에만 표시했습니다.')
for path in [comparison_csv, comparison_md, tidy_csv, interp_md]:
    print_save_status(path)

print('\n9. 확인 필요 사항')
for note in need_check:
    print('-', note)